# Part 2: Predicting Housing Prices in Cook County



## Introduction

In Part 1, we performed some basic Exploratory Data Analysis (EDA), laying out the thought process that may lead to certain modeling decisions. Then, we added a few new features to the dataset and cleaned the data in the process.

In this part, we will specify and fit a linear model to a few features of the housing data to predict house prices. Then, we will analyze the error of the model and brainstorm ways to improve the model's performance. 

## The CCAO Dataset

We'll work with the dataset from the Cook County Assessor's Office (CCAO) in Illinois. This government institution determines property taxes across most of Chicago's metropolitan areas and nearby suburbs. In the United States, all property owners must pay property taxes, which are then used to fund public services, including education, road maintenance, and sanitation. These property tax assessments are based on property values estimated using statistical models considering multiple factors, such as real estate value and construction cost.

While our simple model explains some of the variability in price, there is certainly still a lot of room for improvement —— one reason is we have been only using 1 or 2 features (out of a total of 70+)! In the next part, you will engineer and incorporate more features to improve the model's fairness and accuracy.


# Building the Model

It is time to build our model. Next, we will conduct feature engineering on your training data using the `feature_engine_final` function, and fit and evaluate the model with the training data.


In [1]:
# Import all the necessary libraries

import numpy as np
import pandas as pd
from pandas.api.types import CategoricalDtype

%matplotlib inline
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn import linear_model as lm
from sklearn.model_selection import KFold

import warnings
warnings.filterwarnings("ignore")

import zipfile
import os

from sklearn.preprocessing import OneHotEncoder

In [2]:
training_data = pd.read_csv('cook_county_train.csv')




## Helper Functions

 

In [3]:
# Define additional helper functions
def rmse(predicted, actual):
    return np.sqrt(np.mean((actual - predicted)**2))


def get_outlier_bounds(data, column):
    quartiles = data[column].quantile([0.25, 0.75])
    IQR = quartiles[0.75] - quartiles[0.25]
    return [quartiles[0.25] - 1.5 * IQR, quartiles[0.75] + 5 * IQR]

def remove_outliers(data, variable, lower=-np.inf, upper=np.inf):
    return data[(data[variable] > lower) & (data[variable] < upper)]

def add_total_bathrooms(data):
    with_rooms = data.copy()
    with_rooms['Bathrooms'] = with_rooms['Description'].str.extract(r'(\d+\.\d+) of which are bathrooms').astype('float')
    return with_rooms


def good_condition(data):
    data['in_good_condition'] = data['Repair Condition'].apply(lambda x: 1 if x == 3 else 0)
    return data

def good_basement(data):
    data['has_good_basement'] = data['Basement'].apply(lambda x: 1 if x in [1, 3] else 0)
    return data

def add_indicators(data, column):
    ohe = OneHotEncoder(drop = 'first')
    ohe.fit(data[[column]])
    ohe_columns = pd.DataFrame(ohe.transform(data[[column]]).toarray(), columns = ohe.get_feature_names_out(), index = data.index)
    return data.merge(ohe_columns, left_index=True, right_index=True)




##  Pipeline Function


The function `feature_engine_final` will be our pipeline for feature engineering. It will return X, the feature matrix of predictors, and Y, the outcome for the training data. It will also allow the parameter `is_test_set` to be set to true to return only X, the feature matrix for an unlabeled test dataset so the model can be evaluated on unseen data elsewhere.

In [4]:
def feature_engine_final(data, is_test_set=False):
    if not is_test_set:
        # Processing for the training set (i.e. not the test set)
        # Involves references to sale price
        # Involves filtering certain rows or removing outliers
        data['Log Sale Price'] = np.log(data['Sale Price'])
        data = remove_outliers(data, 'Sale Price', lower = 499)
        data = remove_outliers(data, 'Sale Price', upper = 10000000)


    # Processing for both test and training set
    # Does not involve references to sale price
    # Does not involve removing any rows

    data = add_total_bathrooms(data) #Add bathrooms column
    data['Log Building Square Feet'] = np.log(data['Building Square Feet']) #Log transform building square ft
    data = add_indicators(data, 'Basement')
    data = add_indicators(data, 'Repair Condition')
    data = add_indicators(data, 'Basement Finish')
    data = add_indicators(data, 'Roof Material')
    data = add_indicators(data, 'Property Class')

    data['Estimate'] = data['Estimate (Land)'] + data['Estimate (Building)']
    data['Adj Estimate'] = data['Estimate'] + 0.001
    data['Log Estimate'] = np.log(data['Adj Estimate'])
    

    # Return predictors (X) and response (Y) variables separately
    if is_test_set:
        # Predictors 
        X = data[['Bathrooms',
                  'Log Building Square Feet',
                  'Fireplaces',
                  'Central Air',
                  'Basement_2.0',
                  'Basement_3.0', 
                  'Basement_4.0',
                  'Repair Condition_2.0',
                  'Repair Condition_3.0',
                  'Basement Finish_3.0',
                  'Roof Material_2.0',
                  'Roof Material_3.0',
                  'Roof Material_4.0',
                  'Roof Material_5.0',
                  'Roof Material_6.0',
                  'Log Estimate'
                 ]]
        return X
    else:
        # Predictors. X will not include Log Sale Price
        X = data[['Bathrooms',
                  'Log Building Square Feet',
                  'Fireplaces',
                  'Central Air',
                  'Basement_2.0',
                  'Basement_3.0', 
                  'Basement_4.0',
                  'Repair Condition_2.0',
                  'Repair Condition_3.0',
                  'Basement Finish_3.0',
                  'Roof Material_2.0',
                  'Roof Material_3.0',
                  'Roof Material_4.0',
                  'Roof Material_5.0',
                  'Roof Material_6.0',
                  'Log Estimate'
                 ]]
        # Response variable
        Y = data['Log Sale Price']
        
        return X, Y


## Model Evaluation

To evaluate the model, we'll use 10-fold cross validation on our training data and compute RMSE. Since our model is set up to predict `Log Sale Price`, we'll get our predictions in log scale, then scale them back up to compute RMSE in terms of `Sale Price` as opposed to `Log Sale Price`.

In [5]:
X, Y = feature_engine_final(training_data)

model = lm.LinearRegression(fit_intercept=True)

kf = KFold(n_splits=10)

cv_error = []
for train_idx, valid_idx in kf.split(X):
    # Split the data
    split_X_train, split_X_valid = X.iloc[train_idx], X.iloc[valid_idx]
    split_Y_train, split_Y_valid = Y.iloc[train_idx], Y.iloc[valid_idx]

    # Fit the model on the training split
    model.fit(split_X_train, split_Y_train)
    
    # Compute the RMSE on the validation split
    eval_df = pd.DataFrame({
        'Predicted': model.predict(split_X_valid),
        'Actual': split_Y_valid
    })
    
    eval_df['delog_predicted'] = np.exp(eval_df['Predicted'])
    eval_df['delog_actual'] = np.exp(eval_df['Actual'])

    cv_error.append(rmse(eval_df['delog_predicted'], eval_df['delog_actual']))

print(f'10-fold cross validated RMSE: ${np.round(np.mean(cv_error), 2)}')

10-fold cross validated RMSE: $183615.92
